In [1]:
"""
CS781 Phase 4 - long-context base model
=======================================
Longformer-style SPARSE attention (sliding window + global tokens) with
MLA (Multi-head Latent Attention), pretrained from scratch with masked-LM
on the SAME Hugging Face corpus and the SAME byte-level BPE as Phase 1.

What changed vs. the Phase-1 Model 2 (full causal GQA, 1024 ctx):
  * attention : full O(T^2) GQA  ->  sliding-window + global, O(T*w), MLA (low-rank KV latent
                + decoupled RoPE).  Cost is linear in length, so 4096-token training and
                16k-40k-token fine-tuning/inference are cheap.
  * objective : causal LM  ->  bidirectional masked-LM (the end task is document classification,
                and Longformer-style global attention only makes sense for a bidirectional model)
  * data      : cached 1024-token windows (first 8 windows/doc, 5 epochs)  ->  STREAMED straight from
                Hugging Face, cleaned with the same pipeline as the BPE notebook, whole documents
                packed into 4096-token sequences. N_EPOCHS epochs over a fixed EPOCH_TOKENS-token slice of
                the Phase-1 source mixture 10/50/35/5 (enforced by token count). MLM masks are re-drawn
                every time, so a repeated epoch is not an identical training signal.
  * schedule  : EPOCH based (no wall-clock budget): warmup + cosine decay are spread over
                N_EPOCHS * steps_per_epoch optimizer steps. An ETA is logged; resumable across 12h sessions.
  * tracking  : running train loss + validation loss (fixed held-out docs, fixed masks) every
                EVALS_PER_EPOCH steps and at every epoch end -> train_log.csv, epoch_summary.json,
                loss_curves.png; backbone_best.pt = lowest val loss, backbone_final.pt = last weights.

Design notes
  * Every packed sequence is  [CLS] doc [EOS] [CLS] doc ...  CLS tokens are the GLOBAL tokens.
    A global token attends to (and is attended by) every token of its OWN document only;
    all other attention is a +-ATTN_BLOCK window inside the same document.  Same code path
    handles a single 40k-token document (one CLS at position 0) at fine-tune time.
  * Local attention uses RoPE (relative distance <= window, always inside the training range).
    Global attention is content-only (no RoPE term), so it does not break when the document is
    10x longer than the 4096 tokens seen in pretraining.
  * CLS / MASK are two NEW ids appended after the BPE vocab (tokenizer file is untouched).

Run (Kaggle, GPU on, Internet on):   put the Kaggle secret "HF" (Hugging Face token), then run this file.
Resume after a session ends:         add the previous run's output as an input of the next run, run again.
Put tokenizer.json (+meta.json) from your BPE dataset as an input - if it is missing the script trains
the same BPE recipe from the HF data (slower, and NOT identical to the Phase-1 tokenizer).
"""
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import csv
import gc
import json
import math
import time
import zlib
import random
import hashlib
import shutil
import itertools
import unicodedata
import contextlib
from collections import Counter, deque
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
import torch.utils.data
from huggingface_hub import hf_hub_download, login

# CONFIG
MODEL_TAG = "longformer_mla_mlm_v2"

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = "/kaggle/working/cs781_phase4_longformer_mla" if IS_KAGGLE else "./cs781_phase4_longformer_mla"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
INPUT_ROOT = "/kaggle/input"
TOKENIZER_PATH = None                 # set to a tokenizer.json path to skip the search

# ---- data: same HF repo / files / mixture as the BPE notebook ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
}
SOURCE_RATIO = {"supreme_court": 0.10, "high_courts": 0.50, "tribunals": 0.35, "acts": 0.05}
HF_TOKEN_FALLBACK = ""                # do NOT paste a token here; use the Kaggle secret "HF"

SEQ_LEN = 4096                        # pretraining length (multiple of ATTN_BLOCK)
MIN_DOC_CHARS = 500                   # quality filter (same as BPE notebook)
MAX_DOC_CHARS = 600_000               # keep whole long judgments (BPE notebook spliced at 100k)
MIN_DOC_TOKENS = 256                  # shorter docs are skipped (keeps #CLS per sequence <= GLOBAL_MAX)
MIN_TAIL_TOKENS = 64                  # a document remainder shorter than this is dropped at a sequence cut
VAL_MOD = 40                          # a document is validation if crc32("source:idx") % VAL_MOD == 0
VAL_SEQS = 96                         # validation sequences (~0.4M tokens)
SHUFFLE_SEQS = 1024                   # shuffle buffer (sequences) inside the data worker

# ---- model (~115M parameters) ----
D_MODEL = 768
N_LAYERS = 12
N_HEADS = 12
FFN_HIDDEN = 2560
Q_LORA = 384                          # MLA query latent
KV_LORA = 192                         # MLA key/value latent (the compressed "KV cache" width)
QK_NOPE = 48                          # per-head content dims
QK_ROPE = 16                          # per-head decoupled RoPE dims (k_rope is shared by all heads)
V_DIM = 64
ATTN_BLOCK = 256                      # window = +-256 tokens (513 keys), implemented with 256-token blocks
GLOBAL_MAX = 32                       # max global (CLS) tokens handled per sequence
ROPE_BASE = 10000.0
DROPOUT = 0.0                         # one pass over the data -> no dropout needed

# ---- optimisation ----
N_EPOCHS = 5                          # 4-5 passes; the LR schedule is spread over ALL of them
EPOCH_TOKENS = 80_000_000             # tokens in ONE epoch (a fixed slice of the 10/50/35/5 mixture); total = N_EPOCHS * this
REWIND_EACH_EPOCH = True              # True : every epoch re-reads the SAME slice (true epochs, train-vs-val gap is meaningful)
                                      # False: epochs just continue the stream (always fresh data, epochs = eval checkpoints)
MAX_SESSION_H = 11.3                  # stop cleanly (after saving) before Kaggle's 12h wall; run again to resume
TOKENS_PER_STEP = 32 * SEQ_LEN        # 131k tokens / optimizer step
PEAK_LR = 4e-4
MIN_LR_RATIO = 0.1
WARMUP_FRAC = 0.03                    # warmup = 3% of all optimizer steps (at least 100)
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.98)
GRAD_CLIP = 1.0
MLM_PROB = 0.20                       # 80% [MASK] / 10% random / 10% unchanged
SEED = 1234

LOG_EVERY = 20                        # optimizer steps
EVALS_PER_EPOCH = 5                   # validation runs per epoch (the last one is exactly at the epoch end)
RECENT_STEPS = 50                     # optimizer steps averaged for the "recent train loss" printed next to val loss
SAVE_EVERY_MIN = 30                   # minutes between latest.pt saves
RUN_SELF_TEST = True                  # sparse attention vs dense reference, runs once at start (seconds)

AMP = {"enabled": False, "dtype": torch.float16}


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


# small helpers
def walk_find(filename, roots):
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or HF_TOKEN_FALLBACK or None


def hf_login():
    tok = hf_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        login(token=tok)
        log("Hugging Face login successful.")
    else:
        log("warning: no HF token found (set the Kaggle secret 'HF'); trying anonymous access")
    return tok


def download_sources():
    paths = {}
    tok = hf_token()
    for name, fn in SOURCES.items():
        p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fn, repo_type="dataset", token=tok)
        log(f"{name:14s}: {p}  ({os.path.getsize(p) / 1024 ** 3:.2f} GB)")
        paths[name] = p
    return paths


# text cleaning + dedup  (same pipeline as the BPE notebook)
ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
MULTISPACE = re.compile(r"[ \t]+")
MULTINEWLINE = re.compile(r"\n{3,}")
PAGE_NUMBER = re.compile(r"^\s*(?:page\s*)?\d{1,5}\s*$", re.IGNORECASE)
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE)
DIGIT = re.compile(r"[0-9]")
LETTER = re.compile(r"[^\W\d_]", re.UNICODE)


def normalize_unicode(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = ZERO_WIDTH.sub("", text)
    text = CONTROL_CHARS.sub(" ", text)
    return text.replace("\ufffd", " ")


def fix_line_breaks(text):
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    return re.sub(r"(?<=\w)\n(?=\w)", " ", text)


def normalize_whitespace(text):
    text = MULTISPACE.sub(" ", text)
    text = MULTINEWLINE.sub("\n\n", text)
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)
    return text.strip()


def clean_lines(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if PAGE_NUMBER.match(line):
            continue
        if URL_RE.fullmatch(line) or EMAIL_RE.fullmatch(line):
            continue
        if re.fullmatch(r"[-_=*~.#]{5,}", line):
            continue
        out.append(line)
    return "\n".join(out)


def remove_repeated_lines(text):
    lines = text.splitlines()
    if len(lines) < 10:
        return text
    counts = Counter(l.strip() for l in lines if len(l.strip()) >= 8)
    keep = []
    for l in lines:
        s = l.strip()
        if len(s) >= 15 and counts[s] >= 5 and len(lines) >= 30:
            continue
        keep.append(l)
    return "\n".join(keep)


def is_quality_document(text):
    n = len(text)
    if n < MIN_DOC_CHARS:
        return False
    if text.count("\ufffd") > 10:
        return False
    letters = len(LETTER.findall(text))
    digits = len(DIGIT.findall(text))
    punct = sum(unicodedata.category(c).startswith("P") for c in text)
    space = sum(c.isspace() for c in text)
    if n > 1000 and punct / n > 0.35:
        return False
    if letters < 100:
        return False
    non_space = n - space
    if non_space > 0:
        unusual = non_space - (letters + digits + punct)
        if unusual / non_space > 0.25:
            return False
    if re.search(r"(.)\1{12,}", text):
        return False
    toks = text.split()
    if len(toks) >= 50 and sum(len(x) <= 1 for x in toks) / len(toks) > 0.60:
        return False
    return True


def clean_document(raw):
    if not isinstance(raw, str):
        return None
    text = normalize_unicode(raw)
    text = fix_line_breaks(text)
    text = clean_lines(text)
    text = remove_repeated_lines(text)
    text = normalize_whitespace(text)
    text = MULTISPACE.sub(" ", text).strip()
    if not is_quality_document(text):
        return None
    return text[:MAX_DOC_CHARS]


def extract_text(record):
    if isinstance(record, str):
        return record
    if not isinstance(record, dict):
        return None
    if isinstance(record.get("text"), str):
        return record["text"]
    for key in ("content", "document", "body", "judgment", "article"):
        if isinstance(record.get(key), str):
            return record[key]
    return None


def normalized_hash(text):
    norm = re.sub(r"\s+", " ", text.lower()).strip()
    return hashlib.sha256(norm.encode("utf-8", errors="ignore")).digest()


def simhash(text, max_words=2000):
    """64-bit word-3-shingle SimHash of the first max_words words (numpy-vectorised)."""
    words = text.lower().split()[:max_words]
    if len(words) < 20:
        return None
    hs = np.fromiter(
        (int.from_bytes(hashlib.blake2b(" ".join(words[i:i + 3]).encode("utf-8"), digest_size=8).digest(), "little")
         for i in range(len(words) - 2)), dtype=np.uint64)
    bits = ((hs[:, None] >> np.arange(64, dtype=np.uint64)[None, :]) & np.uint64(1)).astype(np.int64)
    vec = bits.sum(0) * 2 - len(hs)
    out = 0
    for b in range(64):
        if vec[b] >= 0:
            out |= (1 << b)
    return out


class Deduplicator:
    """exact + near duplicate filter (same rule as the BPE notebook: hamming <= 3 inside a 12-bit bucket)"""

    def __init__(self):
        self.exact, self.buckets = set(), {}

    def accept(self, text):
        h = normalized_hash(text)
        if h in self.exact:
            return False
        self.exact.add(h)
        sh = simhash(text)
        if sh is not None:
            cand = self.buckets.setdefault(sh >> 52, [])
            for old in cand:
                if bin(sh ^ old).count("1") <= 3:
                    return False
            cand.append(sh)
        return True


# streaming a huge JSON file (array or dict of records) without loading it
def stream_file_chunks(path, chunk_size=16 * 1024 * 1024):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            yield chunk


def stream_json_records(path):
    decoder = json.JSONDecoder()
    chunks = iter(stream_file_chunks(path))
    st = {"buf": "", "pos": 0, "eof": False}

    def refill():
        try:
            chunk = next(chunks)
        except StopIteration:
            st["eof"] = True
            return
        st["buf"] = st["buf"][st["pos"]:] + chunk
        st["pos"] = 0

    def skip_ws():
        while True:
            while st["pos"] < len(st["buf"]) and st["buf"][st["pos"]].isspace():
                st["pos"] += 1
            if st["pos"] < len(st["buf"]) or st["eof"]:
                return
            refill()

    def parse():
        while True:
            try:
                obj, end = decoder.raw_decode(st["buf"], st["pos"])
                st["pos"] = end
                return obj
            except json.JSONDecodeError:
                if st["eof"]:
                    raise
                refill()

    skip_ws()
    if st["pos"] >= len(st["buf"]):
        return
    first = st["buf"][st["pos"]]
    if first == "[":
        st["pos"] += 1
        while True:
            skip_ws()
            if st["pos"] >= len(st["buf"]) or st["buf"][st["pos"]] == "]":
                return
            yield parse()
            skip_ws()
            if st["pos"] < len(st["buf"]) and st["buf"][st["pos"]] == ",":
                st["pos"] += 1
    elif first == "{":
        st["pos"] += 1
        while True:
            skip_ws()
            if st["pos"] >= len(st["buf"]) or st["buf"][st["pos"]] == "}":
                return
            parse()                                   # key
            skip_ws()
            if st["buf"][st["pos"]] != ":":
                raise ValueError("Malformed JSON: expected ':'")
            st["pos"] += 1
            skip_ws()
            yield parse()
            skip_ws()
            if st["pos"] < len(st["buf"]) and st["buf"][st["pos"]] == ",":
                st["pos"] += 1
    else:
        raise ValueError("Unsupported JSON format")


class DocSource:
    """one HF source file -> endless stream of cleaned documents; position is resumable"""

    def __init__(self, name, path, pos=0, epoch=0):
        self.name, self.path, self.pos, self.epoch = name, path, pos, epoch

    def cleaned(self, role, dedup=None):
        """role 'train' skips validation docs, role 'val' yields only validation docs (single pass)"""
        while True:
            yielded, start_pos = 0, self.pos
            for idx, rec in enumerate(stream_json_records(self.path)):
                if idx < self.pos:                    # fast skip when resuming
                    continue
                self.pos = idx + 1
                is_val = zlib.crc32(f"{self.name}:{idx}".encode()) % VAL_MOD == 0
                if is_val != (role == "val"):
                    continue
                raw = extract_text(rec)
                if raw is None:
                    continue
                text = clean_document(raw)
                if text is None:
                    continue
                if dedup is not None and self.epoch == 0 and not dedup.accept(text):
                    continue
                yielded += 1
                yield text
            if role == "val":
                return
            if yielded == 0 and start_pos == 0:
                raise RuntimeError(f"source {self.name} produced no usable documents")
            self.epoch += 1
            self.pos = 0
            log(f"[data] source {self.name} exhausted -> starting pass {self.epoch + 1} over it (no dedup in extra passes)")


# tokenizer (same byte-level BPE), plus two new ids: CLS and MASK
class LegalByteBPE:
    def __init__(self, tokenizer_json):
        from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_file(tokenizer_json)
        self.base_vocab = self.tokenizer.get_vocab_size()
        vocab = self.tokenizer.get_vocab()
        self.pad_id, self.unk_id, self.eos_id = vocab["<|pad|>"], vocab["<|unk|>"], vocab["<|endoftext|>"]
        self.cls_id = self.base_vocab           # new id
        self.mask_id = self.base_vocab + 1      # new id
        self.model_vocab = ((self.base_vocab + 2 + 63) // 64) * 64    # embedding rows, tensor-core friendly

    def encode(self, text):
        return self.tokenizer.encode(text).ids


def train_bpe_fallback(paths, out_dir, vocab_size=32_000, max_chars=700_000_000):
    """Only used when no tokenizer.json is attached: same recipe as the BPE notebook, on a subset of the HF data."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
    log("=" * 80)
    log("WARNING: tokenizer.json not found -> training a fresh byte-level BPE from the HF data.")
    log("It follows the BPE-notebook recipe but is NOT the same file as your Phase-1 tokenizer.")
    log("=" * 80)
    tk = Tokenizer(models.BPE(unk_token="<|unk|>"))
    tk.normalizer = normalizers.NFKC()
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                  special_tokens=["<|pad|>", "<|unk|>", "<|endoftext|>"],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
    srcs = {n: DocSource(n, paths[n]) for n in SOURCES}
    gens = {n: srcs[n].cleaned("train", Deduplicator()) for n in SOURCES}
    got = {n: 0 for n in SOURCES}

    def it():
        total = 0
        while total < max_chars:
            n = max(SOURCE_RATIO, key=lambda k: SOURCE_RATIO[k] * (total + 1) - got[k])
            t = next(gens[n])[:100_000]
            got[n] += len(t)
            total += len(t)
            yield t

    t0 = time.time()
    tk.train_from_iterator(it(), trainer)
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "tokenizer.json")
    tk.save(path)
    log(f"fallback BPE trained in {(time.time() - t0) / 60:.1f} min, vocab {tk.get_vocab_size()} -> {path}")
    return path


def load_tokenizer(paths):
    """Priority: tokenizer stored with the checkpoint being resumed > TOKENIZER_PATH > a tokenizer.json from the
    attached BPE dataset (byte-level one preferred) > train the same recipe from the HF data (fallback)."""
    path = None
    prev = find_previous()
    if prev is not None and os.path.exists(os.path.join(prev[0], "tokenizer.json")):
        path = os.path.join(prev[0], "tokenizer.json")
        log("using the tokenizer saved with the checkpoint that is being resumed")
    if path is None and TOKENIZER_PATH:
        path = TOKENIZER_PATH
    if path is None:
        cands = set()
        for root in (INPUT_ROOT, os.getcwd(), WORK_DIR):
            for d in walk_find("tokenizer.json", [root]):
                meta = read_json(os.path.join(d, "meta.json")) or {}
                good = meta.get("tokenizer_type") == "true_utf8_byte_level_bpe"
                cands.add((0 if good else 1, len(d), os.path.join(d, "tokenizer.json")))
        cands = sorted(cands)
        if len(cands) > 1:
            log("several tokenizer.json files found (set TOKENIZER_PATH to force one):")
            for c in cands:
                log(f"   {c[2]}")
        path = cands[0][2] if cands else train_bpe_fallback(paths, os.path.join(WORK_DIR, "tokenizer_fallback"))
    os.makedirs(CKPT_DIR, exist_ok=True)
    dst = os.path.join(CKPT_DIR, "tokenizer.json")           # travels with the checkpoints -> same tokenizer next session
    if os.path.abspath(path) != os.path.abspath(dst):
        shutil.copy(path, dst)
    log(f"tokenizer: {path}")
    tok = LegalByteBPE(dst)
    log(f"vocab={tok.base_vocab} (+CLS={tok.cls_id}, MASK={tok.mask_id}) -> embedding rows {tok.model_vocab} | "
        f"pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")
    return tok, dst


# packing: whole documents -> 4096-token sequences  [CLS] doc [EOS] [CLS] doc ...
def pack_docs(doc_iter, L, cls_id, eos_id):
    """doc_iter yields np arrays of token ids; yields (L,) uint16 sequences. Every sequence starts with CLS;
    a document cut at a sequence boundary continues in the next sequence behind a fresh CLS."""
    cur, n = [], 0
    for ids in doc_iter:
        piece = np.concatenate(([cls_id], ids, [eos_id])).astype(np.uint16)
        while len(piece):
            space = L - n
            if len(piece) <= space:
                cur.append(piece)
                n += len(piece)
                piece = piece[:0]
            else:
                cur.append(piece[:space])
                n = L
                rest = piece[space:]
                piece = np.concatenate(([cls_id], rest)).astype(np.uint16) if len(rest) >= MIN_TAIL_TOKENS else rest[:0]
            if n == L:
                yield np.concatenate(cur)
                cur, n = [], 0


class SeqProducer:
    """mixes the 4 sources by TOKEN count (10/50/35/5), cleans + dedups + tokenizes + packs"""

    def __init__(self, paths, tok_path, state, cls_id, eos_id):
        from tokenizers import Tokenizer
        self.tk = Tokenizer.from_file(tok_path)
        self.cls_id, self.eos_id = cls_id, eos_id
        self.dedup = Deduplicator()
        self.src, self.tokens, self.gens = {}, {}, {}
        for n in SOURCES:
            s = state.get(n, {})
            self.src[n] = DocSource(n, paths[n], pos=s.get("pos", 0), epoch=s.get("epoch", 0))
            self.tokens[n] = s.get("tokens", 0)
            self.gens[n] = self.src[n].cleaned("train", self.dedup)

    def snapshot(self):
        return {n: {"pos": self.src[n].pos, "epoch": self.src[n].epoch, "tokens": self.tokens[n]} for n in SOURCES}

    def _docs(self):
        while True:
            total = sum(self.tokens.values()) + 1
            name = max(SOURCE_RATIO, key=lambda k: SOURCE_RATIO[k] * total - self.tokens[k])
            text = next(self.gens[name])
            ids = self.tk.encode(text).ids
            if len(ids) < MIN_DOC_TOKENS:
                continue
            self.tokens[name] += len(ids)
            yield np.asarray(ids, dtype=np.uint16)

    def sequences(self):
        return pack_docs(self._docs(), SEQ_LEN, self.cls_id, self.eos_id)


class PackedStream(torch.utils.data.IterableDataset):
    """runs inside ONE dataloader worker process (cleaning/tokenizing never competes with the GPU loop).
    n_batches=None -> endless; otherwise stops after exactly n_batches micro-batches (= the rest of one epoch)"""

    def __init__(self, paths, tok_path, state, micro_bs, cls_id, eos_id, seed, n_batches=None):
        super().__init__()
        self.paths, self.tok_path, self.state = paths, tok_path, state
        self.micro_bs, self.cls_id, self.eos_id, self.seed = micro_bs, cls_id, eos_id, seed
        self.n_batches = n_batches

    def __iter__(self):
        prod = SeqProducer(self.paths, self.tok_path, self.state, self.cls_id, self.eos_id)
        rng = random.Random(self.seed)
        gen = prod.sequences()
        buf = [next(gen) for _ in range(SHUFFLE_SEQS)]
        sent = 0
        while self.n_batches is None or sent < self.n_batches:
            batch = []
            for _ in range(self.micro_bs):
                j = rng.randrange(len(buf))
                batch.append(buf[j])
                buf[j] = next(gen)
            yield torch.from_numpy(np.stack(batch).astype(np.int32)), prod.snapshot()
            sent += 1


def build_val_tokens(paths, tok):
    """fixed validation sequences from the held-out documents (crc32 split), cached on disk"""
    sig = {"seq": SEQ_LEN, "n": VAL_SEQS, "vocab": tok.base_vocab, "min_tokens": MIN_DOC_TOKENS, "mod": VAL_MOD}
    for d in walk_find("val_manifest.json", [CKPT_DIR, INPUT_ROOT]):
        man = read_json(os.path.join(d, "val_manifest.json"))
        if man and man.get("sig") == sig and os.path.exists(os.path.join(d, "val_tokens.npy")):
            arr = np.load(os.path.join(d, "val_tokens.npy"))
            log(f"loaded cached validation set {arr.shape} from {d}")
            if os.path.abspath(d) != os.path.abspath(CKPT_DIR):
                os.makedirs(CKPT_DIR, exist_ok=True)
                np.save(os.path.join(CKPT_DIR, "val_tokens.npy"), arr)
                atomic_json({"sig": sig}, os.path.join(CKPT_DIR, "val_manifest.json"))
            return arr
    log("building validation set from held-out documents ...")
    docs = []
    for n in SOURCES:
        quota = SOURCE_RATIO[n] * VAL_SEQS * SEQ_LEN * 1.3
        got = 0
        for text in DocSource(n, paths[n]).cleaned("val"):
            ids = tok.encode(text)
            if len(ids) < MIN_DOC_TOKENS:
                continue
            docs.append(np.asarray(ids, dtype=np.uint16))
            got += len(ids)
            if got >= quota:
                break
    random.Random(SEED).shuffle(docs)
    seqs = list(itertools.islice(pack_docs(iter(docs), SEQ_LEN, tok.cls_id, tok.eos_id), VAL_SEQS))
    if len(seqs) < 8:
        raise RuntimeError("could not build a validation set")
    arr = np.stack(seqs)
    os.makedirs(CKPT_DIR, exist_ok=True)
    np.save(os.path.join(CKPT_DIR, "val_tokens.npy"), arr)
    atomic_json({"sig": sig}, os.path.join(CKPT_DIR, "val_manifest.json"))
    log(f"validation set: {arr.shape} ({arr.size / 1e6:.2f}M tokens)")
    return arr


# model
@dataclass
class ModelConfig:
    vocab_size: int = 32064               # embedding rows (>= base_vocab + 2)
    base_vocab: int = 32000
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    cls_id: int = 32000
    mask_id: int = 32001
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    ffn_hidden: int = FFN_HIDDEN
    q_lora: int = Q_LORA
    kv_lora: int = KV_LORA
    qk_nope: int = QK_NOPE
    qk_rope: int = QK_ROPE
    v_dim: int = V_DIM
    attn_block: int = ATTN_BLOCK
    global_max: int = GLOBAL_MAX
    rope_base: float = ROPE_BASE
    dropout: float = DROPOUT


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def apply_rope(x, cos, sin):
    """x (..., T, heads, dr); cos/sin (1, T, 1, dr/2)"""
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)


class AttnCtx:
    """masks / indices shared by all layers of one forward pass"""
    pass


def build_ctx(ids, cfg, inv_freq):
    """
    ids (B, T) with T % attn_block == 0.  Segments = documents (a segment starts at every CLS).
    Local attention : +-attn_block window, same segment only.
    Global attention: CLS tokens (first global_max per row) <-> every token of their own segment.
    """
    B, T = ids.shape
    BLK, G = cfg.attn_block, cfg.global_max
    dev = ids.device
    nb = T // BLK
    valid = ids != cfg.pad_id
    is_cls = (ids == cfg.cls_id) & valid
    seg = torch.cumsum(is_cls.long(), dim=1).masked_fill(~valid, -1)                  # (B,T)
    segb = seg.view(B, nb, BLK)
    sp = F.pad(segb, (0, 0, 1, 1), value=-1)                                          # (B,nb+2,BLK)
    seg3 = torch.cat([sp[:, 0:nb], sp[:, 1:nb + 1], sp[:, 2:nb + 2]], dim=2)          # (B,nb,3BLK)
    r = torch.arange(BLK, device=dev)
    s = torch.arange(3 * BLK, device=dev)
    band = (s[None, :] - BLK - r[:, None]).abs() <= BLK                                # (BLK,3BLK)
    local = band[None, None] & (segb[:, :, :, None] == seg3[:, :, None, :])           # (B,nb,BLK,3BLK)

    ar = torch.arange(T, device=dev)
    key = ar[None, :].expand(B, T).masked_fill(~is_cls, T)
    gpos = key.sort(dim=1).values[:, :G]                                              # (B,G) CLS positions, T = unused
    if gpos.size(1) < G:                                                              # T < G (only in tiny tests)
        gpos = F.pad(gpos, (0, G - gpos.size(1)), value=T)
    gvalid = gpos < T
    gpos_c = gpos.clamp(max=T - 1)
    gseg = seg.gather(1, gpos_c).masked_fill(~gvalid, -2)                             # (B,G)
    posq = ar.view(1, nb, BLK)
    dist = (posq[:, :, :, None] - gpos_c[:, None, None, :]).abs()                      # (B,nb,BLK,G)
    gmask = (segb[:, :, :, None] == gseg[:, None, None, :]) & (dist > BLK) & gvalid[:, None, None, :]
    mask = torch.cat([local, gmask], dim=3).reshape(B * nb, 1, BLK, 3 * BLK + G)      # bool, True = may attend
    gq = ((gseg[:, :, None] == seg[:, None, :]) | (~gvalid)[:, :, None])[:, None]      # (B,1,G,T)
    gscatter = torch.where(gvalid, gpos, torch.full_like(gpos, T))                    # unused slots -> dummy column T
    is_g = torch.zeros(B, T + 1, dtype=torch.long, device=dev).scatter(1, gscatter, torch.ones_like(gscatter))
    is_g = is_g[:, :T] > 0

    ang = ar.float()[:, None] * inv_freq[None, :]
    c = AttnCtx()
    c.B, c.T, c.nb = B, T, nb
    c.cos, c.sin = ang.cos()[None, :, None, :], ang.sin()[None, :, None, :]
    c.seg, c.mask, c.gq, c.gpos_c, c.gscatter, c.is_g = seg, mask, gq, gpos_c, gscatter, is_g
    return c


class MLASparseAttention(nn.Module):
    """Multi-head Latent Attention with sliding-window + global sparsity"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.h, self.dn, self.dr, self.dv = cfg.n_heads, cfg.qk_nope, cfg.qk_rope, cfg.v_dim
        self.dc = cfg.kv_lora
        self.scale = (self.dn + self.dr) ** -0.5
        self.q_down = nn.Linear(cfg.d_model, cfg.q_lora, bias=False)
        self.q_norm = RMSNorm(cfg.q_lora)
        self.q_up = nn.Linear(cfg.q_lora, self.h * (self.dn + self.dr), bias=False)
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora + self.dr, bias=False)       # latent + shared rope key
        self.kv_norm = RMSNorm(cfg.kv_lora)
        self.kv_up = nn.Linear(cfg.kv_lora, self.h * (self.dn + self.dv), bias=False)
        self.o_proj = nn.Linear(self.h * self.dv, cfg.d_model, bias=False)

    def _project(self, x, ctx):
        B, T, _ = x.shape
        q = self.q_up(self.q_norm(self.q_down(x))).view(B, T, self.h, self.dn + self.dr)
        q_nope, q_rope = q.split([self.dn, self.dr], dim=-1)
        ckv, k_rope = self.kv_down(x).split([self.dc, self.dr], dim=-1)
        kv = self.kv_up(self.kv_norm(ckv)).view(B, T, self.h, self.dn + self.dv)
        k_nope, v = kv.split([self.dn, self.dv], dim=-1)
        q_rope = apply_rope(q_rope, ctx.cos, ctx.sin)                                    # (B,T,H,dr)
        k_rope = apply_rope(k_rope[:, :, None, :], ctx.cos, ctx.sin)                     # (B,T,1,dr) shared by all heads
        return q_nope, q_rope, k_nope, k_rope, v

    def forward(self, x, ctx):
        B, T, _ = x.shape
        H, dn, dr, dv = self.h, self.dn, self.dr, self.dv
        BLK, G, nb = self.cfg.attn_block, self.cfg.global_max, ctx.nb
        dq = dn + dr
        q_nope, q_rope, k_nope, k_rope, v = self._project(x, ctx)
        qL = torch.cat([q_nope, q_rope], -1).transpose(1, 2)                              # (B,H,T,dq)
        kL = torch.cat([k_nope, k_rope.expand(-1, -1, H, -1)], -1).transpose(1, 2)
        vL = v.transpose(1, 2)                                                            # (B,H,T,dv)
        kn = k_nope.transpose(1, 2)                                                       # (B,H,T,dn)

        qb = qL.reshape(B, H, nb, BLK, dq)
        kp = F.pad(kL.reshape(B, H, nb, BLK, dq), (0, 0, 0, 0, 1, 1))
        vp = F.pad(vL.reshape(B, H, nb, BLK, dv), (0, 0, 0, 0, 1, 1))
        k3 = torch.cat([kp[:, :, 0:nb], kp[:, :, 1:nb + 1], kp[:, :, 2:nb + 2]], dim=3)   # (B,H,nb,3BLK,dq)
        v3 = torch.cat([vp[:, :, 0:nb], vp[:, :, 1:nb + 1], vp[:, :, 2:nb + 2]], dim=3)

        gi = ctx.gpos_c                                                                   # (B,G)
        kg = kn.gather(2, gi[:, None, :, None].expand(B, H, G, dn))                       # (B,H,G,dn)
        kg = F.pad(kg, (0, dr))                                                           # no RoPE term for global keys
        vg = vL.gather(2, gi[:, None, :, None].expand(B, H, G, dv))
        k_all = torch.cat([k3, kg[:, :, None].expand(B, H, nb, G, dq)], dim=3)            # (B,H,nb,3BLK+G,dq)
        v_all = torch.cat([v3, vg[:, :, None].expand(B, H, nb, G, dv)], dim=3)

        def to4(t):                                                                       # (B,H,nb,L,d) -> (B*nb,H,L,d)
            return t.permute(0, 2, 1, 3, 4).reshape(B * nb, H, t.size(3), t.size(4))

        out = F.scaled_dot_product_attention(to4(qb), to4(k_all), to4(v_all), attn_mask=ctx.mask, scale=self.scale)
        out = out.reshape(B, nb, H, BLK, dv).permute(0, 2, 1, 3, 4).reshape(B, H, T, dv)

        # global tokens attend to every token of their own document (content-only, no RoPE)
        qg = q_nope.transpose(1, 2).gather(2, gi[:, None, :, None].expand(B, H, G, dn))
        og = F.scaled_dot_product_attention(qg, kn, vL, attn_mask=ctx.gq, scale=self.scale)   # (B,H,G,dv)
        ext = F.pad(out, (0, 0, 0, 1))                                                    # dummy slot T for unused globals
        ext = ext.scatter(2, ctx.gscatter[:, None, :, None].expand(B, H, G, dv), og.to(ext.dtype))
        out = ext[:, :, :T]
        return self.o_proj(out.transpose(1, 2).reshape(B, T, H * dv))

    def forward_dense(self, x, ctx):
        """slow O(T^2) reference with the same sparsity pattern (used by the self-test only)"""
        B, T, _ = x.shape
        qn, qr, kn, kr, v = self._project(x, ctx)
        s_n = torch.einsum("bihd,bjhd->bhij", qn.float(), kn.float())
        s_r = torch.einsum("bihd,bjd->bhij", qr.float(), kr[:, :, 0].float())
        ar = torch.arange(T, device=x.device)
        band = (ar[:, None] - ar[None, :]).abs() <= self.cfg.attn_block
        use_rope = band[None, None] & (~ctx.is_g)[:, None, :, None]
        sc = (s_n + torch.where(use_rope, s_r, torch.zeros_like(s_r))) * self.scale
        same = ctx.seg[:, :, None] == ctx.seg[:, None, :]
        allowed = same[:, None] & (band[None, None] | ctx.is_g[:, None, None, :] | ctx.is_g[:, None, :, None])
        p = sc.masked_fill(~allowed, float("-inf")).softmax(-1)
        out = torch.einsum("bhij,bjhd->bihd", p, v.float())
        return self.o_proj(out.reshape(B, T, self.h * self.dv).to(x.dtype))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn, self.ffn = MLASparseAttention(cfg), SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, ctx):
        x = x + self.drop(self.attn(self.norm1(x), ctx))
        return x + self.drop(self.ffn(self.norm2(x)))


class LongformerMLA(nn.Module):
    """bidirectional sparse-attention encoder (MLM head tied to the embedding)"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        inv = 1.0 / (cfg.rope_base ** (torch.arange(0, cfg.qk_rope, 2).float() / cfg.qk_rope))
        self.register_buffer("inv_freq", inv, persistent=False)
        self.use_ckpt = False
        self.apply(self._init)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, ids):
        """ids (B,T) -> hidden (B,T,D). T is right-padded to a multiple of attn_block with pad_id."""
        B, T0 = ids.shape
        BLK = self.cfg.attn_block
        if T0 % BLK:
            ids = F.pad(ids, (0, BLK - T0 % BLK), value=self.cfg.pad_id)
        ctx = build_ctx(ids, self.cfg, self.inv_freq)
        x = self.embed(ids)
        for blk in self.blocks:
            if self.use_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(blk, x, ctx, use_reentrant=False)
            else:
                x = blk(x, ctx)
        return self.final_norm(x)[:, :T0]

    def mlm_loss(self, inp, labels):
        h = self.encode(inp)
        sel = labels != -100
        logits = F.linear(h[sel], self.embed.weight)                  # logits only at masked positions
        return F.cross_entropy(logits.float(), labels[sel])


class LongDocBinaryClassifier(nn.Module):
    """for the 0/1 task: input ids = [CLS] doc tokens ... [EOS] (pad_id padded). Returns one logit per doc."""

    def __init__(self, backbone, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        d = backbone.cfg.d_model
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d, 1))

    def forward(self, ids):
        h = self.backbone.encode(ids)
        valid = (ids != self.backbone.cfg.pad_id).unsqueeze(-1).to(h.dtype)
        mean = (h * valid).sum(1) / valid.sum(1).clamp(min=1)
        return self.head(torch.cat([h[:, 0], mean], dim=-1)).squeeze(-1)


def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model = LongformerMLA(ModelConfig(**ck["config"]))
    model.load_state_dict(ck["model"])
    return model.to(device)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# MLM masking
def mlm_mask(ids, cfg, gen=None):
    elig = (ids != cfg.pad_id) & (ids != cfg.cls_id) & (ids != cfg.eos_id)
    r = torch.rand(ids.shape, device=ids.device, generator=gen)
    sel = elig & (r < MLM_PROB)
    labels = ids.masked_fill(~sel, -100)
    r2 = torch.rand(ids.shape, device=ids.device, generator=gen)
    inp = ids.clone()
    inp[sel & (r2 < 0.8)] = cfg.mask_id
    rnd = sel & (r2 >= 0.8) & (r2 < 0.9)
    rand_tok = torch.randint(0, cfg.base_vocab, ids.shape, device=ids.device, generator=gen)
    inp[rnd] = rand_tok[rnd]
    return inp, labels


@torch.no_grad()
def evaluate(model, val_tokens, device, bs):
    model.eval()
    cfg = model.cfg
    gen = torch.Generator().manual_seed(SEED)                                  # same masks every evaluation
    total, count = 0.0, 0
    for i in range(0, len(val_tokens), bs):
        ids = torch.from_numpy(val_tokens[i:i + bs].astype(np.int64))
        inp, labels = mlm_mask(ids, cfg, gen)
        inp, labels = inp.to(device), labels.to(device)
        with amp_ctx():
            h = model.encode(inp)
            sel = labels != -100
            logits = F.linear(h[sel], model.embed.weight)
        total += F.cross_entropy(logits.float(), labels[sel], reduction="sum").item()
        count += int(sel.sum().item())
    model.train()
    return total / max(count, 1)


# self test + micro-batch autotune
@torch.no_grad()
def attention_selftest(device):
    """sparse (block) attention must equal the dense reference with the same pattern (fp32, tiny shapes)"""
    cfg = ModelConfig(vocab_size=128, base_vocab=100, cls_id=100, mask_id=101, d_model=48, n_heads=3, q_lora=24,
                      kv_lora=16, qk_nope=8, qk_rope=8, v_dim=8, attn_block=8, global_max=4, n_layers=2, ffn_hidden=64)
    torch.manual_seed(0)
    attn = MLASparseAttention(cfg).to(device).float()
    for p in attn.parameters():
        nn.init.normal_(p, std=0.3) if p.ndim == 2 else None
    inv = (1.0 / (cfg.rope_base ** (torch.arange(0, cfg.qk_rope, 2).float() / cfg.qk_rope))).to(device)
    B, T = 3, 64
    ids = torch.randint(3, 90, (B, T), device=device)
    for b, ps in enumerate([[0, 20, 37], [0, 9, 60], [0]]):
        ids[b, ps] = cfg.cls_id
    ids[1, 50:] = cfg.pad_id
    ctx = build_ctx(ids, cfg, inv)
    x = torch.randn(B, T, cfg.d_model, device=device)
    a, d = attn(x, ctx), attn.forward_dense(x, ctx)
    ok = (ids != cfg.pad_id).unsqueeze(-1)
    err = ((a - d).abs() * ok).max().item()
    if not err < 1e-4:
        raise RuntimeError(f"sparse attention self-test FAILED: max diff {err:.3e}")
    log(f"self-test passed: block-sparse MLA attention == dense reference (max diff {err:.2e})")


def autotune_batch(model, cfg, device):
    """largest micro-batch that fits (optionally with activation checkpointing); returns (use_ckpt, micro_bs)"""
    total_mem = torch.cuda.get_device_properties(0).total_memory
    adam_bytes = 8 * count_params(model)
    model.train()
    for use_ckpt, mb in [(False, 8), (False, 4), (False, 2), (True, 8), (True, 4), (True, 2), (True, 1)]:
        model.use_ckpt = use_ckpt
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            ids = torch.randint(3, cfg.base_vocab, (mb, SEQ_LEN), device=device)
            ids[:, ::1024] = cfg.cls_id
            inp, labels = mlm_mask(ids, cfg)
            with amp_ctx():
                loss = model.mlm_loss(inp, labels)
            loss.backward()
            peak = torch.cuda.max_memory_allocated()
            model.zero_grad(set_to_none=True)
            del loss, ids, inp, labels
            fits = peak + adam_bytes < 0.90 * total_mem
            log(f"  autotune: micro_bs={mb} grad_ckpt={use_ckpt} peak={peak / 1024 ** 3:.1f} GB "
                f"(+{adam_bytes / 1024 ** 3:.1f} GB Adam) of {total_mem / 1024 ** 3:.1f} GB -> {'OK' if fits else 'too big'}")
            if fits:
                return use_ckpt, mb
        except torch.cuda.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            gc.collect()
            torch.cuda.empty_cache()
            log(f"  autotune: micro_bs={mb} grad_ckpt={use_ckpt} -> out of memory")
    raise RuntimeError("model does not fit on this GPU even with micro_bs=1 + checkpointing")


# optimisation helpers + checkpoints
def lr_at(step, total_steps, warmup):
    """linear warmup, then cosine decay to MIN_LR_RATIO * PEAK_LR exactly at the last step of the last epoch"""
    if step < warmup:
        return PEAK_LR * (step + 1) / warmup
    p = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return PEAK_LR * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p)))


def make_optimizer(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (decay if p.ndim >= 2 and "embed" not in name else no_decay).append(p)
    groups = [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=PEAK_LR, betas=BETAS, eps=1e-8, fused=torch.cuda.is_available())


def find_previous():
    best = None
    for d in walk_find("latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "latest_meta.json"))
        if meta is None or meta.get("tag") != MODEL_TAG or not os.path.exists(os.path.join(d, "latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over(src_dir):
    if os.path.abspath(src_dir) == os.path.abspath(CKPT_DIR):
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    for fn in ("backbone_best.pt", "best_meta.json", "train_log.csv", "val_tokens.npy", "val_manifest.json"):
        s, dst = os.path.join(src_dir, fn), os.path.join(CKPT_DIR, fn)
        if os.path.exists(s) and not os.path.exists(dst):
            shutil.copy(s, dst + ".tmp")
            os.replace(dst + ".tmp", dst)


LOG_FIELDS = ["time", "kind", "epoch", "global_step", "train_loss", "val_loss", "val_ppl", "lr", "tokens_B", "tok_per_s", "train_h"]


def append_log(row):
    path = os.path.join(CKPT_DIR, "train_log.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new:
            wr.writeheader()
        wr.writerow({k: row.get(k, "") for k in LOG_FIELDS})


def plot_curves(steps_per_epoch):
    """train / val loss curves -> loss_curves.png (a plotting problem must never stop training)"""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        path = os.path.join(CKPT_DIR, "train_log.csv")
        if not os.path.exists(path):
            return
        tr, va = [], []
        with open(path) as f:
            for r in csv.DictReader(f):
                try:
                    step = int(r["global_step"])
                    if r["kind"] == "train" and r["train_loss"]:
                        tr.append((step, float(r["train_loss"])))
                    elif r["kind"] == "val" and r["val_loss"]:
                        va.append((step, float(r["val_loss"])))
                except (ValueError, KeyError, TypeError):
                    continue
        if not tr and not va:
            return
        fig, ax = plt.subplots(figsize=(10, 5.5))
        if tr:
            ax.plot(*zip(*tr), lw=1, alpha=0.85, label="train loss (mean of each log window)")
        if va:
            ax.plot(*zip(*va), "o-", lw=2, ms=4, label="val loss")
        for e in range(1, N_EPOCHS):
            ax.axvline(e * steps_per_epoch, ls="--", c="gray", lw=0.8)
        allv = np.array([v for _, v in tr + va])
        lo = float(allv.min()) * 0.97
        ax.set_ylim(lo, max(float(np.percentile(allv, 90)), lo * 1.1))      # hide the huge first steps
        ax.set_xlabel("optimizer step   (dashed lines = epoch boundaries)")
        ax.set_ylabel("masked-LM loss")
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(CKPT_DIR, "loss_curves.png"), dpi=110)
        plt.close(fig)
    except Exception as e:                                                   # noqa
        log(f"  (loss plot skipped: {e})")


# training loop (epoch based, resumable)
def train(model, cfg, tok_path, paths, val_tokens, device, t_session):
    os.makedirs(CKPT_DIR, exist_ok=True)
    use_ckpt, mb = autotune_batch(model, cfg, device)
    model.use_ckpt = use_ckpt
    accum = TOKENS_PER_STEP // (mb * SEQ_LEN)
    tok_per_step = mb * accum * SEQ_LEN
    steps_per_epoch = max(1, round(EPOCH_TOKENS / tok_per_step))
    total_steps = steps_per_epoch * N_EPOCHS
    warmup = max(100, int(WARMUP_FRAC * total_steps))
    eval_every = max(1, steps_per_epoch // EVALS_PER_EPOCH)
    log(f"micro_bs={mb} x accum={accum} -> {tok_per_step:,} tokens/step | grad_ckpt={use_ckpt}")
    log(f"plan: {N_EPOCHS} epochs x {steps_per_epoch} steps ({steps_per_epoch * tok_per_step / 1e6:.0f}M tokens/epoch) = "
        f"{total_steps} steps, {total_steps * tok_per_step / 1e6:.0f}M tokens | warmup {warmup} steps | "
        f"validation every {eval_every} steps | "
        f"{'SAME data slice every epoch' if REWIND_EACH_EPOCH else 'fresh data every epoch'}")

    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"global_step": 0, "train_seconds": 0.0, "tokens_seen": 0, "best_val": float("inf"), "data": {},
             "epoch": 0, "epoch_steps": 0, "epoch_loss_sum": 0.0, "epoch_loss_n": 0, "history": []}

    prev = find_previous()
    if prev is not None:
        d, meta = prev
        log(f"resuming from {d}: step {meta['global_step']}, trained {fmt_time(meta['train_seconds'])}")
        ck = torch.load(os.path.join(d, "latest.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        state = {k: ck.get(k, state[k]) for k in state}
        del ck
        carry_over(d)
    else:
        log("no previous checkpoint, starting from scratch")
    best_meta = read_json(os.path.join(CKPT_DIR, "best_meta.json"))
    if best_meta:
        state["best_val"] = min(state["best_val"], best_meta["val_loss"])
    if state["epoch"] >= N_EPOCHS:
        log("all epochs were already finished in an earlier session")
        return True
    log(f"position: epoch {state['epoch'] + 1}/{N_EPOCHS}, step {state['epoch_steps']}/{steps_per_epoch} of this epoch")

    def save_latest():
        atomic_torch_save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                           **state, "config": asdict(cfg), "tag": MODEL_TAG}, os.path.join(CKPT_DIR, "latest.pt"))
        atomic_json({"global_step": state["global_step"], "train_seconds": state["train_seconds"],
                     "tokens_seen": state["tokens_seen"], "epoch": state["epoch"], "epoch_steps": state["epoch_steps"],
                     "tag": MODEL_TAG, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
                    os.path.join(CKPT_DIR, "latest_meta.json"))
        log(f"  >> saved latest.pt (epoch {state['epoch'] + 1}/{N_EPOCHS}, step {state['global_step']}, "
            f"{state['tokens_seen'] / 1e6:.0f}M tokens, trained {fmt_time(state['train_seconds'])})")

    def save_backbone(name, extra=None):
        atomic_torch_save({"model": model.state_dict(), "config": asdict(cfg), "tag": MODEL_TAG, **(extra or {})},
                          os.path.join(CKPT_DIR, name))

    recent = deque(maxlen=RECENT_STEPS)               # recent per-step train losses (shown next to every val loss)
    cur = {"val": float("nan")}

    def run_validation():
        vl = evaluate(model, val_tokens, device, mb)
        tr = sum(recent) / len(recent) if recent else float("nan")
        improved = vl < state["best_val"]
        if improved:
            state["best_val"] = vl
            save_backbone("backbone_best.pt", {"val_loss": vl, "global_step": state["global_step"],
                                               "epoch": state["epoch"] + 1})
            atomic_json({"tag": MODEL_TAG, "val_loss": vl, "val_ppl": math.exp(vl), "global_step": state["global_step"],
                         "epoch": state["epoch"] + 1}, os.path.join(CKPT_DIR, "best_meta.json"))
        log(f"  >> VAL epoch {state['epoch'] + 1}/{N_EPOCHS} step {state['global_step']} | "
            f"train(last {len(recent)} steps) {tr:.4f} | val {vl:.4f} (ppl {math.exp(vl):.2f}) | gap {vl - tr:+.4f} | "
            f"{'NEW BEST -> backbone_best.pt' if improved else 'best %.4f' % state['best_val']}")
        append_log({"time": time.strftime("%H:%M:%S"), "kind": "val", "epoch": state["epoch"] + 1,
                    "global_step": state["global_step"], "train_loss": f"{tr:.4f}", "val_loss": f"{vl:.4f}",
                    "val_ppl": f"{math.exp(vl):.2f}", "tokens_B": f"{state['tokens_seen'] / 1e9:.3f}",
                    "train_h": f"{state['train_seconds'] / 3600:.2f}"})
        cur["val"] = vl
        plot_curves(steps_per_epoch)
        model.train()
        return vl, tr

    model.train()
    t_prev = t_save = time.time()
    t_log, tok_log, loss_log, n_log = time.time(), 0, 0.0, 0
    tps_ema, gnorm, bad_steps = None, float("nan"), 0

    for epoch in range(state["epoch"], N_EPOCHS):
        remaining = steps_per_epoch - state["epoch_steps"]
        it = loader = None
        if remaining > 0:
            log(f"=== epoch {epoch + 1}/{N_EPOCHS}: {remaining} steps to go | starting the data worker "
                f"(filling the shuffle buffer can take a few minutes) ...")
            loader = torch.utils.data.DataLoader(
                PackedStream(paths, tok_path, state["data"], mb, cfg.cls_id, cfg.eos_id,
                             SEED + 7919 * epoch + state["epoch_steps"], n_batches=remaining * accum),
                batch_size=None, num_workers=1, prefetch_factor=6)
            it = iter(loader)
            first = next(it)
            it = itertools.chain([first], it)
            t_prev = t_log = time.time()                       # buffer fill is not training time
            tok_log, loss_log, n_log = 0, 0.0, 0

        while state["epoch_steps"] < steps_per_epoch:
            if time.time() - t_session > MAX_SESSION_H * 3600:
                save_latest()
                log("session time limit reached; run again in a new session (attach this output as an input) to continue.")
                return False
            lr = lr_at(state["global_step"], total_steps, warmup)
            for g in opt.param_groups:
                g["lr"] = lr
            step_loss = torch.zeros((), device=device)
            for _ in range(accum):
                ids, snap = next(it)
                ids = ids.to(device, non_blocking=True).long()
                inp, labels = mlm_mask(ids, cfg)
                with amp_ctx():
                    loss = model.mlm_loss(inp, labels)
                scaler.scale(loss / accum).backward()
                step_loss += loss.detach() / accum
                state["data"] = snap
            scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            state["global_step"] += 1
            state["epoch_steps"] += 1
            state["tokens_seen"] += tok_per_step
            tok_log += tok_per_step
            sl = step_loss.item()
            if math.isfinite(sl):
                bad_steps = 0
                loss_log += sl
                n_log += 1
                recent.append(sl)
                state["epoch_loss_sum"] += sl
                state["epoch_loss_n"] += 1
            else:
                bad_steps += 1
                if bad_steps >= 25:                            # do not burn GPU hours on a diverged run
                    raise RuntimeError("training loss has been NaN/inf for 25 steps in a row - stopping. "
                                       "latest.pt / backbone_best.pt are from before the divergence.")
            now = time.time()
            state["train_seconds"] += now - t_prev
            t_prev = now

            if state["global_step"] % LOG_EVERY == 0:
                avg = loss_log / max(n_log, 1)
                tps = tok_log / max(now - t_log, 1e-9)
                tps_ema = tps if tps_ema is None else 0.9 * tps_ema + 0.1 * tps
                eta_s = (total_steps - state["global_step"]) * tok_per_step / max(tps_ema, 1.0)
                log(f"ep {epoch + 1}/{N_EPOCHS} step {state['epoch_steps']}/{steps_per_epoch} "
                    f"(global {state['global_step']}/{total_steps}) | train {avg:.4f} | val {cur['val']:.4f} | "
                    f"best {state['best_val']:.4f} | gnorm {float(gnorm):.2f} | lr {lr:.2e} | {tps:,.0f} tok/s | "
                    f"{state['tokens_seen'] / 1e6:.0f}M tok | trained {fmt_time(state['train_seconds'])} | "
                    f"ETA {fmt_time(eta_s)} (~{(state['train_seconds'] + eta_s) / 3600:.1f} h total)")
                atomic_json({"epoch": epoch + 1, "epochs": N_EPOCHS, "global_step": state["global_step"],
                             "total_steps": total_steps, "train_loss": avg, "last_val_loss": cur["val"],
                             "best_val": state["best_val"], "tokens_M": state["tokens_seen"] / 1e6,
                             "eta_h": round(eta_s / 3600, 2), "updated": time.strftime("%Y-%m-%d %H:%M:%S")},
                            os.path.join(CKPT_DIR, "progress.json"))
                append_log({"time": time.strftime("%H:%M:%S"), "kind": "train", "epoch": epoch + 1,
                            "global_step": state["global_step"], "train_loss": f"{avg:.4f}", "lr": f"{lr:.3e}",
                            "tokens_B": f"{state['tokens_seen'] / 1e9:.3f}", "tok_per_s": f"{tps:.0f}",
                            "train_h": f"{state['train_seconds'] / 3600:.2f}"})
                t_log, tok_log, loss_log, n_log = now, 0, 0.0, 0

            if state["epoch_steps"] % eval_every == 0 and state["epoch_steps"] < steps_per_epoch:
                run_validation()
            if time.time() - t_save > SAVE_EVERY_MIN * 60:
                save_latest()
                t_save = time.time()

        # ---------------- epoch finished ----------------
        vl, tr = run_validation()
        ep_mean = state["epoch_loss_sum"] / max(state["epoch_loss_n"], 1)
        state["history"].append({"epoch": epoch + 1, "train_loss_epoch_mean": round(ep_mean, 4),
                                 "train_loss_last_steps": round(tr, 4), "val_loss": round(vl, 4),
                                 "val_ppl": round(math.exp(vl), 2), "global_step": state["global_step"],
                                 "tokens_M": round(state["tokens_seen"] / 1e6, 1),
                                 "train_h": round(state["train_seconds"] / 3600, 2)})
        atomic_json(state["history"], os.path.join(CKPT_DIR, "epoch_summary.json"))
        log(f"=== epoch {epoch + 1}/{N_EPOCHS} finished | mean train loss {ep_mean:.4f} | val loss {vl:.4f} "
            f"(ppl {math.exp(vl):.2f}) | best val {state['best_val']:.4f} | trained {fmt_time(state['train_seconds'])}")
        state["epoch"] = epoch + 1
        state["epoch_steps"] = 0
        state["epoch_loss_sum"], state["epoch_loss_n"] = 0.0, 0
        if REWIND_EACH_EPOCH:
            state["data"] = {}                                 # next epoch re-reads the same slice from the start
        save_latest()
        t_save = time.time()
        del it, loader
        gc.collect()

    log("training finished: all epochs done")
    save_backbone("backbone_final.pt", {"val_loss": cur["val"], "best_val": state["best_val"],
                                        "global_step": state["global_step"]})
    plot_curves(steps_per_epoch)
    log("epoch | mean train loss | val loss | val ppl")
    for h in state["history"]:
        log(f"  {h['epoch']:>3} | {h['train_loss_epoch_mean']:.4f} | {h['val_loss']:.4f} | {h['val_ppl']:.2f}")
    log(f"final backbone (last weights): {os.path.join(CKPT_DIR, 'backbone_final.pt')} | "
        f"lowest-val weights: {os.path.join(CKPT_DIR, 'backbone_best.pt')} "
        f"(load with load_backbone(path); use LongDocBinaryClassifier for the 0/1 task)")
    return True


# main
def main():
    t_session = time.time()
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)
    os.makedirs(CKPT_DIR, exist_ok=True)
    hf_login()

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible - this script is meant for a GPU runtime")

    if RUN_SELF_TEST:
        attention_selftest(device)

    log("downloading source files from Hugging Face (cached after the first time) ...")
    paths = download_sources()
    tok, tok_path = load_tokenizer(paths)
    val_tokens = build_val_tokens(paths, tok)

    cfg = ModelConfig(vocab_size=tok.model_vocab, base_vocab=tok.base_vocab, pad_id=tok.pad_id, unk_id=tok.unk_id,
                      eos_id=tok.eos_id, cls_id=tok.cls_id, mask_id=tok.mask_id)
    assert SEQ_LEN % cfg.attn_block == 0
    model = LongformerMLA(cfg).to(device)
    log(f"model parameters: {count_params(model) / 1e6:.1f}M | window +-{cfg.attn_block} | global tokens/seq <= {cfg.global_max}")

    train(model, cfg, tok_path, paths, val_tokens, device, t_session)
    log("done.")


if __name__ == "__main__":
    main()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
GPU: Tesla T4 | autocast dtype: torch.float16
self-test passed: block-sparse MLA attention == dense reference (max diff 1.73e-06)
downloading source files from Hugging Face (cached after the first time) ...


Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

supreme_court : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json  (1.23 GB)


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

... (4 outputs left out)

tribunals     : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json  (6.04 GB)


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

acts          : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json  (0.68 GB)
tokenizer: /kaggle/input/datasets/krishkalg/bpe-prep/cs781_clean_bpe32k/tokenizer.json
vocab=32000 (+CLS=32000, MASK=32001) -> embedding rows 32064 | pad=0 unk=1 eos=2
building validation set from held-out documents ...
validation set: (96, 4096) (0.39M tokens)
model parameters: 114.6M | window +-256 | global tokens/seq <= 32
  autotune: micro_bs=8 grad_ckpt=False -> out of memory
  autotune: micro_bs=4 grad_ckpt=False peak=13.8 GB (+0.9 GB Adam) of 14.6 GB -> too big
  autotune: micro_bs=2 grad_ckpt=False peak=7.2 GB (+0.9 GB Adam) of 14.6 GB -> OK
micro_bs=2 x accum=16 -> 131,072 tokens/step | grad_ckpt=False
plan: 5 epochs x 610 steps (80M tokens/epoch) = 3050 steps, 400M tokens | warmup 100 steps | validation every 122 steps | SAME data slice every epoch
no previous checkpoint, starting from s